# Feature Characterization

Here I group the features by how similar they quantify image textures.

Below I created functions that would produce the GLCM that maximize/minimize each feature function. From here we can group the the features by the resulting GLCMs. Some features are maximized by:
- Distrubutions concentrated at the bottom corner ($1,1$, $N,N$ entry)
- Concentration at the main-diagonal off-diagonal or cross-diagonal
- Concentration at the off diagonals right above and below the main-diagonal
- Concentration at opposite (cross) corners
- Uniform distribution


In [49]:
import numpy as np
from scipy.optimize import minimize

In [46]:
def maximize_haralick(feature_func, n_levels):
    def objective(flat_p):
        P = flat_p.reshape((n_levels, n_levels))
        return -feature_func(P)

    cons = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bnds = [(0, 1) for _ in range(n_levels**2)]
    init_guess = np.ones(n_levels**2) / (n_levels**2)
    
    res = minimize(objective, init_guess, method='SLSQP', bounds=bnds, constraints=cons)
    return res.x.reshape((n_levels, n_levels)), -res.fun


def minimize_haralick(feature_func, n_levels):
    def objective(flat_p):
        P = flat_p.reshape((n_levels, n_levels))
        return feature_func(P)

    cons = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bnds = [(0, 1) for _ in range(n_levels**2)]
    init_guess = np.ones(n_levels**2) / (n_levels**2)
    
    res = minimize(objective, init_guess, method='SLSQP', bounds=bnds, constraints=cons)
    return res.x.reshape((n_levels, n_levels)), res.fun

In [47]:
def get_stats(P):
    N = P.shape[0]
    # Ensure P is normalized (crucial for optimizer stability)
    P = P / np.sum(P)
    
    # Grids (1-based for formula consistency)
    i, j = np.indices((N, N))
    i, j = i + 1, j + 1
    
    # Marginal probabilities
    p_x = np.sum(P, axis=1)
    p_y = np.sum(P, axis=0)
    
    # Means
    mu_x = np.sum(np.arange(1, N + 1) * p_x)
    mu_y = np.sum(np.arange(1, N + 1) * p_y)
    mu = (mu_x + mu_y) / 2
    
    # Standard Deviations
    sigma_x = np.sqrt(np.sum(((np.arange(1, N + 1) - mu_x)**2) * p_x))
    sigma_y = np.sqrt(np.sum(((np.arange(1, N + 1) - mu_y)**2) * p_y))
    
    # p_x+y and p_x-y distributions
    p_sum = np.zeros(2 * N + 1)
    p_diff = np.zeros(N)
    for row in range(N):
        for col in range(N):
            p_sum[(row+1) + (col+1)] += P[row, col]
            p_diff[abs(row - col)] += P[row, col]
            
    return {
        'P': P, 'N': N, 'i': i, 'j': j, 
        'mu_x': mu_x, 'mu_y': mu_y, 'mu': mu,
        'sigma_x': sigma_x, 'sigma_y': sigma_y,
        'p_x': p_x, 'p_y': p_y, 'p_sum': p_sum, 'p_diff': p_diff
    }


def feat_autocorr(P):
    s = get_stats(P)
    return np.sum(s['i'] * s['j'] * s['P'])

def feat_contrast(P):
    s = get_stats(P)
    return np.sum(s['p_diff'] * (np.arange(s['N'])**2))

def feat_cluster_prominence(P):
    s = get_stats(P)
    return np.sum(P * (s['i'] + s['j'] - 2 * s['mu'])**4)

def feat_cluster_shade(P):
    s = get_stats(P)
    return np.sum(P * (s['i'] + s['j'] - 2 * s['mu'])**3)

def feat_cluster_tendency(P):
    s = get_stats(P)
    return np.sum(P * (s['i'] + s['j'] - 2 * s['mu'])**2)

def feat_entropy(P):
    # Small epsilon to avoid log(0)
    return -np.sum(P * np.log(P + 1e-15))

def feat_asm(P):
    return np.sum(P**2)

def feat_correlation(P):
    s = get_stats(P)
    if s['sigma_x'] * s['sigma_y'] == 0: return 0
    term = np.sum(s['i'] * s['j'] * P)
    return (term - (s['mu_x'] * s['mu_y'])) / (s['sigma_x'] * s['sigma_y'])

def feat_homogeneity(P):
    s = get_stats(P)
    return np.sum(P / (1 + (s['i'] - s['j'])**2))

def feat_invdiff(P):
    s = get_stats(P)
    return np.sum(P / (1 + np.abs(s['i'] - s['j'])))

def feat_diffavg(P):
    s = get_stats(P)
    diff_matrix = np.abs(s['i'] - s['j'])
    dissimilarity = np.sum(diff_matrix * s['P'])
    return dissimilarity

def feat_diffent(P):
    s = get_stats(P)
    p_diff = s['p_diff']
    p_diff_nonzero = p_diff[p_diff > 1e-15]
    diff_entropy = -np.sum(p_diff_nonzero * np.log(p_diff_nonzero))
    return diff_entropy

def feat_diffvar(P):
    s = get_stats(P)
    p_diff = s['p_diff']
    N = s['N']
    k_values = np.arange(N)
    mu_diff = np.sum(k_values * p_diff)
    diff_variance = np.sum(((k_values - mu_diff)**2) * p_diff)
    return diff_variance

def feat_imc1(P):
    s = get_stats(P)
    eps = 1e-15 
    hxy = -np.sum(s['P'] * np.log(s['P'] + eps))
    hx = -np.sum(s['p_x'] * np.log(s['p_x'] + eps))
    hy = -np.sum(s['p_y'] * np.log(s['p_y'] + eps))
    marginal_product = np.outer(s['p_x'], s['p_y'])
    hxy1 = -np.sum(s['P'] * np.log(marginal_product + eps))
    
    denominator = max(hx, hy)
    if denominator < eps:
        return 0.0
    imc1 = (hxy - hxy1) / denominator
    return imc1

def feat_imc2(P):
    s = get_stats(P)
    eps = 1e-15 
    hxy = -np.sum(s['P'] * np.log(s['P'] + eps))
    hx = -np.sum(s['p_x'] * np.log(s['p_x'] + eps))
    hy = -np.sum(s['p_y'] * np.log(s['p_y'] + eps))
    hxy2 = hx + hy
    
    term = 1.0 - np.exp(-2.0 * (hxy2 - hxy))
    imc2 = np.sqrt(max(0, term))
    return imc2

def feat_jointavg(P):
    s = get_stats(P)
    return np.sum(s['i'] * s['P'])

def feat_sumavg(P):
    s = get_stats(P)
    return np.sum((s['i'] + s['j']) * s['P'])

def feat_sument(P):
    s = get_stats(P)
    p_sum = s['p_sum']
    p_sum_nonzero = p_sum[p_sum > 1e-15]
    return -np.sum(p_sum_nonzero * np.log(p_sum_nonzero))

def feat_sumsqr(P):
    s = get_stats(P)
    return np.sum((s['i'] - s['mu_x'])**2 * s['P'])

def feat_invvar(P):
    s = get_stats(P)
    p_diff = s['p_diff']
    N = s['N']
    k_values = np.arange(1, N)
    relevant_p_diff = p_diff[1:]
    return np.sum(relevant_p_diff / (k_values**2))

In [181]:
import numpy as np
from scipy.optimize import differential_evolution
from skimage.feature import graycomatrix

def _glcm_objective(flat_p, feature_func, n_levels, mode):
    P = flat_p.reshape((n_levels, n_levels))

    # Force Symmetry: Haralick features assume P(i,j) = P(j,i)
    P = (P + P.T) / 2
    
    s = np.sum(P)
    if s <= 0: return 1e10
    
    P_norm = P / s
    val = feature_func(P_norm)
    
    if np.isnan(val) or np.isinf(val):
        return 1e10
        
    return -val if mode == 'max' else val

def optimize_haralick(feature_func, n_levels, mode='max'):
    def get_seeds(n_seeds):
        seeds = []
        for _ in range(n_seeds):
            img = np.random.randint(0, n_levels, (8, 8))
            g = graycomatrix(img, [1], [0], levels=n_levels, symmetric=True, normed=True)
            seeds.append(g.squeeze().flatten())
        return np.array(seeds)

    bounds = [(0, 1) for _ in range(n_levels**2)]
    
    # Population size: 10-15 times the number of variables
    pop_total = 10 * (n_levels**2)
    seeds = get_seeds(pop_total)

    # Note: workers=1 avoids the AttributeError in interactive environments
    res = differential_evolution(
        _glcm_objective,
        bounds,
        args=(feature_func, n_levels, mode),
        init=seeds,
        strategy='best1bin',
        maxiter=500,
        tol=1e-6,
        polish=True, # This uses a local solver to find the exact peak
        workers=1     
    )

    best_P = res.x.reshape((n_levels, n_levels))
    best_P = (best_P + best_P.T) / 2
    best_P /= np.sum(best_P)
    
    return best_P, (-res.fun if mode == 'max' else res.fun)

## Autocorrelation
$$ \sum^{N}_{i=1} \sum^{N}_{j=1} (i \cdot j) p(i, j) $$

A GLCM that would maximize this function, would be one that is concentratwed at the diagonals of greater entries.

lets try to get as many combinations of 8 by 8 images that have the highest concentration in the bottom corner entries.

In [22]:
optimize_haralick(feat_autocorr, n_levels=4, mode='max')

(array([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 1.]]),
 np.float64(16.0))

In [24]:
optimize_haralick(feat_autocorr, n_levels=4, mode='min')

(array([[1., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]]),
 np.float64(1.0))

## Cluster Prominence
$$ \sum_{i=1}^{N} \sum_{j=1}^{N} (i + j - \mu_x - \mu_y)^4 p(i, j) $$

In [138]:
opt_glcm, val = maximize_haralick(feat_cluster_prominence, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 4), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.5 0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0.5]]  
 With 81.0


In [168]:
optimize_haralick(feat_cluster_prominence, n_levels=4, mode='max')

(array([[0.21132489, 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.78867511]]),
 np.float64(107.9999999999989))

In [26]:
optimize_haralick(feat_cluster_prominence, n_levels=4, mode='min')

(array([[0.        , 0.        , 0.        , 0.21215795],
        [0.        , 0.        , 0.28784205, 0.        ],
        [0.        , 0.28784205, 0.        , 0.        ],
        [0.21215795, 0.        , 0.        , 0.        ]]),
 np.float64(6.223015277861142e-61))

In [180]:
_, val = optimize_haralick(feat_cluster_prominence, n_levels=7, mode='min')
np.round(_, 4), val

(array([[0.    , 0.    , 0.    , 0.    , 0.    , 0.    , 0.0761],
        [0.    , 0.    , 0.    , 0.    , 0.    , 0.0762, 0.0805],
        [0.    , 0.    , 0.    , 0.    , 0.0774, 0.075 , 0.    ],
        [0.    , 0.    , 0.    , 0.0406, 0.0945, 0.    , 0.    ],
        [0.    , 0.    , 0.0774, 0.0945, 0.    , 0.    , 0.    ],
        [0.    , 0.0762, 0.075 , 0.    , 0.    , 0.    , 0.    ],
        [0.0761, 0.0805, 0.    , 0.    , 0.    , 0.    , 0.    ]]),
 np.float64(0.06250000001295027))

## Cluster Shade


$$ \sum_{i=1}^{N} \sum_{j=1}^{N} (i + j - \mu_x - \mu_y)^3 p(i, j) $$


In [29]:
optimize_haralick(feat_cluster_shade, n_levels=4, mode='max')

(array([[0.78867514, 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.21132486]]),
 np.float64(20.784609690826517))

In [30]:
optimize_haralick(feat_cluster_shade, n_levels=4, mode='min')

(array([[0.21132113, 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.78867887]]),
 np.float64(-20.784609685600145))

## Cluster Tendency

$$ \sum_{i=1}^{N} \sum_{j=1}^{N} (i + j - \mu_x - \mu_y)^2 p(i, j) $$

In [31]:
optimize_haralick(feat_cluster_tendency, n_levels=4, mode='max')

(array([[0.49999999, 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.50000001]]),
 np.float64(8.999999999999993))

In [32]:
optimize_haralick(feat_cluster_tendency, n_levels=4, mode='min')

(array([[0.        , 0.        , 0.        , 0.21967323],
        [0.        , 0.        , 0.28032677, 0.        ],
        [0.        , 0.28032677, 0.        , 0.        ],
        [0.21967323, 0.        , 0.        , 0.        ]]),
 np.float64(0.0))

In [176]:
_, val = optimize_haralick(feat_cluster_tendency, n_levels=7, mode='min')
np.round(_, 4), val

(array([[0.    , 0.    , 0.    , 0.    , 0.    , 0.    , 0.1433],
        [0.    , 0.    , 0.    , 0.    , 0.    , 0.1433, 0.    ],
        [0.    , 0.    , 0.    , 0.    , 0.1416, 0.    , 0.    ],
        [0.    , 0.    , 0.    , 0.1433, 0.    , 0.    , 0.    ],
        [0.    , 0.    , 0.1416, 0.    , 0.    , 0.    , 0.    ],
        [0.    , 0.1433, 0.    , 0.    , 0.    , 0.    , 0.    ],
        [0.1433, 0.    , 0.    , 0.    , 0.    , 0.    , 0.    ]]),
 np.float64(3.155443620884047e-30))

## Contrast
$$ \sum_{i=1}^{N} \sum_{j=1}^{N} (i - j)^2 p(i, j) $$

In [34]:
optimize_haralick(feat_contrast, n_levels=4, mode='max')

(array([[0. , 0. , 0. , 0.5],
        [0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. ],
        [0.5, 0. , 0. , 0. ]]),
 np.float64(9.0))

In [35]:
optimize_haralick(feat_contrast, n_levels=4, mode='min')

(array([[0.29453782, 0.        , 0.        , 0.        ],
        [0.        , 0.24883718, 0.        , 0.        ],
        [0.        , 0.        , 0.16492798, 0.        ],
        [0.        , 0.        , 0.        , 0.29169703]]),
 np.float64(0.0))

## Correlation
$$ \frac{\sum^{N_g}_{i=1}\sum^{N_g}_{j=1}{p(i,j)ij-\mu_x\mu_y}}{\sigma_x(i)\sigma_y(j)} $$

In [36]:
optimize_haralick(feat_correlation, n_levels=4, mode='max')

(array([[0.35423581, 0.        , 0.        , 0.        ],
        [0.        , 0.14446512, 0.        , 0.        ],
        [0.        , 0.        , 0.16944711, 0.        ],
        [0.        , 0.        , 0.        , 0.33185196]]),
 np.float64(0.9999999999999999))

In [38]:
optimize_haralick(feat_correlation, n_levels=4, mode='min')

(array([[0.        , 0.        , 0.        , 0.28866628],
        [0.        , 0.        , 0.21133372, 0.        ],
        [0.        , 0.21133372, 0.        , 0.        ],
        [0.28866628, 0.        , 0.        , 0.        ]]),
 np.float64(-0.9999999999999999))

## Difference Average 

$$ \sum^{N}_{i=1}\sum^{N}_{j=1}{|i-j|p(i,j)} $$

In [39]:
optimize_haralick(feat_diffavg, n_levels=4, mode='max')

(array([[0. , 0. , 0. , 0.5],
        [0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. ],
        [0.5, 0. , 0. , 0. ]]),
 np.float64(3.0))

In [40]:
optimize_haralick(feat_diffavg, n_levels=4, mode='min')

(array([[0.29038507, 0.        , 0.        , 0.        ],
        [0.        , 0.25945242, 0.        , 0.        ],
        [0.        , 0.        , 0.21915513, 0.        ],
        [0.        , 0.        , 0.        , 0.23100737]]),
 np.float64(0.0))

In [154]:
optimize_haralick(feat_diffavg, n_levels=5, mode='min')

(array([[0.19085505, 0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.15278158, 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.21192326, 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.2217041 , 0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.222736  ]]),
 np.float64(0.0))

## Difference entropy

$$ -\sum_{k=0}^{N-1} p_{x-y}(k) \log p_{x-y}(k) $$

In [41]:
optimize_haralick(feat_diffent, n_levels=4, mode='max')

(array([[0.10971872, 0.01661838, 0.07029125, 0.12499899],
        [0.01661838, 0.07545378, 0.05178657, 0.0547094 ],
        [0.07029125, 0.05178657, 0.01887438, 0.05659513],
        [0.12499899, 0.0547094 , 0.05659513, 0.04595369]]),
 np.float64(1.3862943611076868))

In [151]:
optimize_haralick(feat_diffent, n_levels=4, mode='min')

(array([[0.30312268, 0.        , 0.        , 0.        ],
        [0.        , 0.28897669, 0.        , 0.        ],
        [0.        , 0.        , 0.29696295, 0.        ],
        [0.        , 0.        , 0.        , 0.11093769]]),
 np.float64(-0.0))

In [145]:
_, val = optimize_haralick(feat_diffent, n_levels=7, mode='min')
np.round(_, 4), val

(array([[0.    , 0.0869, 0.    , 0.    , 0.    , 0.    , 0.    ],
        [0.0869, 0.    , 0.0873, 0.    , 0.    , 0.    , 0.    ],
        [0.    , 0.0873, 0.    , 0.0976, 0.    , 0.    , 0.    ],
        [0.    , 0.    , 0.0976, 0.    , 0.0739, 0.    , 0.    ],
        [0.    , 0.    , 0.    , 0.0739, 0.    , 0.0811, 0.    ],
        [0.    , 0.    , 0.    , 0.    , 0.0811, 0.    , 0.0732],
        [0.    , 0.    , 0.    , 0.    , 0.    , 0.0732, 0.    ]]),
 np.float64(-0.0))

## Difference variance

$$ \sum_{k=0}^{N-1} (k - \mu_{x-y})^2 p_{x-y}(k) $$


In [52]:
optimize_haralick(feat_diffvar, n_levels=4, mode='max')

(array([[0.13857769, 0.        , 0.        , 0.25000002],
        [0.        , 0.15402609, 0.        , 0.        ],
        [0.        , 0.        , 0.03946401, 0.        ],
        [0.25000002, 0.        , 0.        , 0.16793217]]),
 np.float64(2.2499999999999867))

In [53]:
optimize_haralick(feat_diffvar, n_levels=4, mode='min')

(array([[1.09049422e-06, 2.22412960e-01, 3.52883609e-07, 9.83517106e-08],
        [2.22412960e-01, 7.65855844e-08, 1.49505258e-01, 5.49781538e-08],
        [3.52883609e-07, 1.49505258e-01, 1.14365624e-06, 1.28080072e-01],
        [9.83517106e-08, 5.49781538e-08, 1.28080072e-01, 9.71713763e-08]]),
 np.float64(4.0104431922041525e-06))

In [142]:
_, val = optimize_haralick(feat_diffvar, n_levels=7, mode='min')
np.round(_, 4), val

(array([[0.    , 0.093 , 0.    , 0.    , 0.    , 0.    , 0.    ],
        [0.093 , 0.    , 0.0597, 0.    , 0.    , 0.    , 0.    ],
        [0.    , 0.0597, 0.    , 0.093 , 0.    , 0.    , 0.    ],
        [0.    , 0.    , 0.093 , 0.    , 0.0894, 0.    , 0.    ],
        [0.    , 0.    , 0.    , 0.0894, 0.    , 0.0719, 0.    ],
        [0.    , 0.    , 0.    , 0.    , 0.0719, 0.    , 0.093 ],
        [0.    , 0.    , 0.    , 0.    , 0.    , 0.093 , 0.    ]]),
 np.float64(0.0))

## Energy
$$ \sum_{i=1}^{N} \sum_{j=1}^{N} p(i, j)^2 $$

In [54]:
optimize_haralick(feat_asm, n_levels=4, mode='max')

(array([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 1.]]),
 np.float64(1.0))

In [132]:
optimize_haralick(feat_asm, n_levels=7, mode='max')

(array([[0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0.5, 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0. , 0.5, 0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0. ]]),
 np.float64(0.5))

In [134]:
optimize_haralick(feat_asm, n_levels=7, mode='max')

(array([[0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 0., 0.]]),
 np.float64(1.0))

In [55]:
optimize_haralick(feat_asm, n_levels=4, mode='min')

(array([[0.0625, 0.0625, 0.0625, 0.0625],
        [0.0625, 0.0625, 0.0625, 0.0625],
        [0.0625, 0.0625, 0.0625, 0.0625],
        [0.0625, 0.0625, 0.0625, 0.0625]]),
 np.float64(0.06250000000000003))

## Entropy
$$ -\sum_{i=1}^{N} \sum_{j=1}^{N} p(i, j) \log p(i, j) $$

In [56]:
optimize_haralick(feat_entropy, n_levels=4, mode='max')

(array([[0.06250088, 0.06249981, 0.06249983, 0.0625001 ],
        [0.06249981, 0.06249983, 0.0624999 , 0.06250016],
        [0.06249983, 0.0624999 , 0.06250022, 0.06249981],
        [0.0625001 , 0.06250016, 0.06249981, 0.06249985]]),
 np.float64(2.7725887222304424))

In [131]:
optimize_haralick(feat_entropy, n_levels=4, mode='min')

(array([[0., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]]),
 np.float64(-1.110223024625156e-15))

In [123]:
optimize_haralick(feat_entropy, n_levels=7, mode='min')

(array([[0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0.5, 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0.5]]),
 np.float64(0.6931471805599433))

In [130]:
optimize_haralick(feat_entropy, n_levels=7, mode='min')

(array([[0.25, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
        [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
        [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
        [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.25],
        [0.  , 0.  , 0.  , 0.  , 0.25, 0.  , 0.  ],
        [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
        [0.  , 0.  , 0.  , 0.25, 0.  , 0.  , 0.  ]]),
 np.float64(1.3862943611198866))

## Homogeneity (2)
$$ \sum_{i=1}^{N} \sum_{j=1}^{N} \frac{p(i, j)}{1 + (i - j)^2} $$

In [59]:
optimize_haralick(feat_homogeneity, n_levels=4, mode='max')

(array([[0.23110889, 0.        , 0.        , 0.        ],
        [0.        , 0.26901721, 0.        , 0.        ],
        [0.        , 0.        , 0.28958188, 0.        ],
        [0.        , 0.        , 0.        , 0.21029201]]),
 np.float64(1.0))

In [60]:
optimize_haralick(feat_homogeneity, n_levels=4, mode='min')

(array([[0. , 0. , 0. , 0.5],
        [0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. ],
        [0.5, 0. , 0. , 0. ]]),
 np.float64(0.1))

## Information measure of correlation 1

$$ \frac{HXY-HXY1}{\max\{HX,HY\}} $$

In [61]:
optimize_haralick(feat_imc1, n_levels=4, mode='max')

(array([[0.12220289, 0.0569303 , 0.07345872, 0.09698373],
        [0.0569303 , 0.026521  , 0.03422419, 0.04517997],
        [0.07345872, 0.03422419, 0.04415364, 0.05829954],
        [0.09698373, 0.04517997, 0.05829954, 0.07696957]]),
 np.float64(-2.8809846297935917e-10))

In [62]:
optimize_haralick(feat_imc1, n_levels=4, mode='min')

(array([[0.        , 0.        , 0.24415666, 0.        ],
        [0.        , 0.22977503, 0.        , 0.        ],
        [0.24415666, 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.28191165]]),
 np.float64(-0.9999999999999941))

In [119]:
optimize_haralick(feat_imc1, n_levels=5, mode='min')

(array([[0.18951095, 0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.20864595, 0.        ],
        [0.        , 0.        , 0.21277707, 0.        , 0.        ],
        [0.        , 0.20864595, 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.18042008]]),
 np.float64(-0.9999999999999906))

## Information measure of correlation 2

$$ \sqrt{1-e^{-2(HXY2-HXY)}} $$

In [63]:
optimize_haralick(feat_imc2, n_levels=4, mode='max')

(array([[0.        , 0.25000303, 0.        , 0.        ],
        [0.25000303, 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.24998997, 0.        ],
        [0.        , 0.        , 0.        , 0.25000397]]),
 np.float64(0.9682458365344558))

In [113]:
optimize_haralick(feat_imc2, n_levels=5, mode='max')

(array([[0.20005946, 0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.20012723, 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.20012279, 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.19984526],
        [0.        , 0.        , 0.        , 0.19984526, 0.        ]]),
 np.float64(0.979795888673091))

In [64]:
optimize_haralick(feat_imc2, n_levels=4, mode='min')

(array([[0.08857641, 0.06577048, 0.06767095, 0.07560004],
        [0.06577048, 0.04883644, 0.05024759, 0.05613517],
        [0.06767095, 0.05024759, 0.05169952, 0.05775722],
        [0.07560004, 0.05613517, 0.05775722, 0.06452471]]),
 np.float64(1.2990531157078367e-07))

## Inverse difference

$$ \sum_{i=1}^{N} \sum_{j=1}^{N} \frac{p(i, j)}{1 + |i - j|} $$

In [65]:
optimize_haralick(feat_invdiff, n_levels=4, mode='max')

(array([[0.29675028, 0.        , 0.        , 0.        ],
        [0.        , 0.08503168, 0.        , 0.        ],
        [0.        , 0.        , 0.31863434, 0.        ],
        [0.        , 0.        , 0.        , 0.29958369]]),
 np.float64(1.0))

In [67]:
optimize_haralick(feat_invdiff, n_levels=4, mode='min')

(array([[0. , 0. , 0. , 0.5],
        [0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. ],
        [0.5, 0. , 0. , 0. ]]),
 np.float64(0.25))

## Joint average

$$ \sum_{i=1}^{N} \sum_{j=1}^{N} i \cdot p(i, j) $$

In [69]:
optimize_haralick(feat_jointavg, n_levels=4, mode='max')

(array([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 1.]]),
 np.float64(4.0))

In [70]:
optimize_haralick(feat_jointavg, n_levels=4, mode='min')

(array([[1., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]]),
 np.float64(1.0))

## Sum average

$$ \sum^{2N}_{k=2}{k \cdot p_{x+y}(k)} $$

In [71]:
optimize_haralick(feat_sumavg, n_levels=4, mode='max')

(array([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 1.]]),
 np.float64(8.0))

In [72]:
optimize_haralick(feat_sumavg, n_levels=4, mode='min')

(array([[1., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]]),
 np.float64(2.0))

## Sum entropy

$$  - \sum^{2N}_{k=2}{p_{x+y}(k)\log p_{x+y}(k)} $$

In [73]:
optimize_haralick(feat_sument, n_levels=4, mode='max')

(array([[0.14285736, 0.07142728, 0.04596971, 0.04622751],
        [0.07142728, 0.05091825, 0.02520189, 0.05034497],
        [0.04596971, 0.02520189, 0.04216544, 0.07142694],
        [0.04622751, 0.05034497, 0.07142694, 0.14286233]]),
 np.float64(1.9459101488790758))

In [106]:
optimize_haralick(feat_sument, n_levels=4, mode='min')

(array([[0.       , 0.       , 0.       , 0.       ],
        [0.       , 0.       , 0.       , 0.3226734],
        [0.       , 0.       , 0.3546532, 0.       ],
        [0.       , 0.3226734, 0.       , 0.       ]]),
 np.float64(1.1102230246251564e-16))

In [107]:
optimize_haralick(feat_sument, n_levels=5, mode='min')

(array([[0.        , 0.        , 0.        , 0.        , 0.21173567],
        [0.        , 0.        , 0.        , 0.20160222, 0.        ],
        [0.        , 0.        , 0.17332422, 0.        , 0.        ],
        [0.        , 0.20160222, 0.        , 0.        , 0.        ],
        [0.21173567, 0.        , 0.        , 0.        , 0.        ]]),
 np.float64(1.1102230246251564e-16))

## Sum of squares

$$ \sum^{N}_{i=1}\sum^{N}_{j=1}{(i-\mu_x)^2p(i,j)} $$

In [75]:
optimize_haralick(feat_sumsqr, n_levels=4, mode='max')

(array([[0.2412292 , 0.        , 0.        , 0.25877082],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.25877082, 0.        , 0.        , 0.24122916]]),
 np.float64(2.249999999999996))

In [77]:
optimize_haralick(feat_sumsqr, n_levels=4, mode='min')

(array([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 0.]]),
 np.float64(0.0))

In [101]:
optimize_haralick(feat_sumsqr, n_levels=7, mode='min')

(array([[0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.]]),
 np.float64(0.0))

## Maximal Correlation Coefficient

$$ \sqrt{\lambda_2 Q(i,j)} $$

$$ Q(i, j) = \displaystyle\sum^{N}_{k=0}{\frac{p(i,k)p(j, k)}{p_x(i)p_y(k)}} $$

In [79]:
def feat_mcc(P):
    s = get_stats(P)
    N = s['N']
    P = s['P']
    px = s['p_x']
    py = s['p_y']
    eps = 1e-15

    Q = np.zeros((N, N))

    px_inv = 1.0 / (px + eps)
    py_inv = 1.0 / (py + eps)
    
    for i in range(N):
        for j in range(N):
            term = (P[i, :] * P[j, :]) * px_inv[i] * py_inv
            Q[i, j] = np.sum(term)

    evals = np.linalg.eigvals(Q)
    
    evals = np.sort(np.real(evals))
    
    if len(evals) >= 2:
        mcc = np.sqrt(max(0, evals[-2]))
    else:
        mcc = 0.0
        
    return mcc

In [80]:
optimize_haralick(feat_mcc, n_levels=4, mode='max')

(array([[1.39586641e-01, 9.99927440e-02, 2.79282435e-08, 2.73678097e-02],
        [9.99927440e-02, 1.76991377e-01, 2.53386275e-09, 7.65789122e-02],
        [2.79282435e-08, 2.53386275e-09, 1.61941680e-01, 4.39585281e-10],
        [2.73678097e-02, 7.65789122e-02, 4.39585281e-10, 1.13601309e-01]]),
 np.float64(0.999999772307211))

In [81]:
optimize_haralick(feat_mcc, n_levels=4, mode='min')

(array([[0.04680808, 0.05256533, 0.06080561, 0.05617273],
        [0.05256533, 0.0590307 , 0.06828451, 0.0630818 ],
        [0.06080561, 0.06828451, 0.07898897, 0.07297068],
        [0.05617273, 0.0630818 , 0.07297068, 0.06741093]]),
 np.float64(2.617340310405584e-08))

## Inverse variance

$$ \sum^{N-1}_{k=1}{\frac{p_{x-y}(k)}{k^2}} $$

In [99]:
optimize_haralick(feat_invvar, n_levels=4, mode='max')

(array([[1.52277879e-07, 1.89601866e-01, 8.34622232e-08, 1.24338315e-07],
        [1.89601866e-01, 2.97604227e-07, 1.50483584e-01, 1.35494669e-07],
        [8.34622232e-08, 1.50483584e-01, 1.04498109e-07, 1.59913841e-01],
        [1.24338315e-07, 1.35494669e-07, 1.59913841e-01, 1.77356861e-07]]),
 np.float64(0.9999987187816926))

In [86]:
optimize_haralick(feat_invvar, n_levels=5, mode='max')

(array([[0.        , 0.14179966, 0.        , 0.        , 0.        ],
        [0.14179966, 0.        , 0.1144132 , 0.        , 0.        ],
        [0.        , 0.1144132 , 0.        , 0.10253782, 0.        ],
        [0.        , 0.        , 0.10253782, 0.        , 0.14124932],
        [0.        , 0.        , 0.        , 0.14124932, 0.        ]]),
 np.float64(1.0))

In [92]:
optimize_haralick(feat_invvar, n_levels=4, mode='min')

(array([[0.24551896, 0.        , 0.        , 0.        ],
        [0.        , 0.27013531, 0.        , 0.        ],
        [0.        , 0.        , 0.24720956, 0.        ],
        [0.        , 0.        , 0.        , 0.23713618]]),
 np.float64(0.0))